In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilează cu pass managers

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Modalitatea recomandată de a transpila un Circuit este să creezi un staged pass manager și apoi să execuți metoda `run` cu circuitul ca intrare. Această pagină explică cum să transpilezi circuite cuantice în acest mod.
## Ce este un (staged) pass manager?
În contextul SDK-ului Qiskit, transpilarea se referă la procesul de transformare a unui Circuit de intrare într-o formă potrivită pentru execuție pe un dispozitiv cuantic. Transpilarea are loc, de obicei, într-o secvență de pași numiți transpiler passes. Circuitul este procesat de fiecare transpiler pass în ordine, iar ieșirea unui pass devine intrarea următorului. De exemplu, un pass ar putea parcurge circuitul și îmbina toate secvențele consecutive de Gate-uri cu un singur Qubit, iar apoi pasul următor ar putea sintetiza aceste Gate-uri în setul de bază al dispozitivului țintă. Transpiler passes incluse în Qiskit se află în modulul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Un pass manager este un obiect care stochează o listă de transpiler passes și le poate executa pe un Circuit. Creează un pass manager prin inițializarea unui [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) cu o listă de transpiler passes. Pentru a rula transpilarea pe un Circuit, apelează metoda [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) cu un Circuit ca intrare.

Un staged pass manager este un tip special de pass manager care reprezintă un nivel de abstractizare superior față de un pass manager obișnuit. În timp ce un pass manager obișnuit este compus din mai multe transpiler passes, un staged pass manager este compus din mai multe *pass managers*. Aceasta este o abstractizare utilă deoarece transpilarea are loc, de obicei, în etape discrete, după cum este descris în [Etapele Transpiler-ului](transpiler-stages), fiecare etapă fiind reprezentată de un pass manager. Staged pass managers sunt reprezentate de clasa [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). Restul acestei pagini descrie cum să creezi și să personalizezi (staged) pass managers.
## Generează un staged pass manager preset
Pentru a crea un staged pass manager preset cu valori implicite rezonabile, folosește funcția [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Pentru a transpila un Circuit sau o listă de circuite cu un pass manager, transmite circuitul sau lista de circuite metodei `run`. Să facem asta pe un Circuit cu doi Qubiți constând dintr-un Hadamard urmat de două Gate-uri CX adiacente:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Consultă [Valori implicite de transpilare și opțiuni de configurare](defaults-and-configuration-options) pentru o descriere a posibilelor argumente ale funcției `generate_preset_pass_manager`. Argumentele pentru `generate_preset_pass_manager` corespund argumentelor funcției [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Ai dificultăți în a-ți aminti detalii despre pass manager? Încearcă să întrebi Qiskit Code Assistant." />

Dacă pass managers preset nu îți satisfac nevoile, personalizează transpilarea creând (staged) pass managers sau chiar transpilation passes. Restul acestei pagini descrie cum să creezi pass managers. Pentru instrucțiuni despre cum să creezi transpilation passes, consultă [Scrie-ți propriul transpiler pass](custom-transpiler-pass).

## Creează propriul tău pass manager

Modulul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) include multe transpiler passes care pot fi folosite pentru a crea pass managers. Pentru a crea un pass manager, inițializează un `PassManager` cu o listă de passes. De exemplu, următorul cod creează un transpiler pass care îmbină Gate-urile adiacente cu doi Qubiți și apoi le sintetizează într-o bază formată din Gate-urile [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) și [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Pentru a demonstra acest pass manager în acțiune, testează-l pe un Circuit cu doi Qubiți constând dintr-un Hadamard urmat de două Gate-uri CX adiacente:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Pentru a rula pass manager-ul pe Circuit, apelează metoda `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Pentru un exemplu mai avansat care arată cum să creezi un pass manager pentru a implementa tehnica de suprimare a erorilor cunoscută sub numele de dynamical decoupling, consultă [Creează un pass manager pentru dynamical decoupling](dynamical-decoupling-pass-manager).

## Creează un staged pass manager

Un `StagedPassManager` este un pass manager compus din etape individuale, unde fiecare etapă este definită de o instanță `PassManager`. Poți crea un `StagedPassManager` specificând etapele dorite. De exemplu, următorul cod creează un staged pass manager cu două etape, `init` și `translation`. Etapa `translation` este definită de pass manager-ul creat anterior.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Nu există o limită pentru numărul de etape pe care le poți adăuga într-un staged pass manager.

O altă modalitate utilă de a crea un staged pass manager este să începi cu un staged pass manager preset și apoi să înlocuiești unele dintre etape. De exemplu, următorul cod generează un pass manager preset cu nivelul de optimizare 3, apoi specifică o etapă `pre_layout` personalizată.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[Funcțiile generator de etape](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) pot fi utile pentru construirea unor pass managers personalizate.
Acestea generează etape care oferă funcționalități comune folosite în multe pass managers.
De exemplu, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) poate fi folosit pentru a genera o etapă
care „înglobează" un `Layout` inițial selectat dintr-un layout pass în dispozitivul țintă specificat.

## Pași următori
> **Tip:** - [Scrie un transpiler pass personalizat](custom-transpiler-pass).
>     - [Creează un pass manager pentru dynamical decoupling](dynamical-decoupling-pass-manager).
>     - Pentru a afla mai multe despre funcția `generate_preset_passmanager`, consultă subiectul [Setări implicite de transpilare și opțiuni de configurare](defaults-and-configuration-options).
>     - Încearcă ghidul [Compară setările Transpiler-ului](/guides/circuit-transpilation-settings).
>     - Consultă [documentația API a Transpiler-ului.](https://docs.quantum.ibm.com/api/qiskit/transpiler)